# UC01 — Asistente Conversacional de Planta (Talk to your Plant)**Objetivo:** Construir un asistente RAG que permita a los operadores de ETAP/EDAR consultar en lenguaje natural el estado de la planta, procedimientos y alarmas.**Tecnologías:** Cortex Search · CORTEX.COMPLETE · Streamlit**Tiempo estimado:** 12 minutos

## Paso 1 — Configuración del EntornoCreamos la base de datos aislada, el esquema y el warehouse para este proyecto.

In [ ]:
CREATE DATABASE IF NOT EXISTS TALK_TO_PLANT_DB;CREATE SCHEMA IF NOT EXISTS TALK_TO_PLANT_DB.PUBLIC;USE DATABASE TALK_TO_PLANT_DB;USE SCHEMA PUBLIC;CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Manuales OperativosGeneramos 80 documentos técnicos simulando manuales, fichas de equipo, protocolos de emergencia y procedimientos operativos de planta de tratamiento de agua.

In [ ]:
CREATE OR REPLACE TABLE manuales_planta ASSELECT    'DOC-' || LPAD(SEQ4()::STRING, 4, '0') AS doc_id,    CASE MOD(SEQ4(), 4)        WHEN 0 THEN 'Procedimiento_Operativo'        WHEN 1 THEN 'Ficha_Equipo'        WHEN 2 THEN 'Protocolo_Emergencia'        ELSE 'Manual_Mantenimiento'    END AS tipo,    CASE MOD(SEQ4(), 6)        WHEN 0 THEN 'Pretratamiento' WHEN 1 THEN 'Coagulacion'        WHEN 2 THEN 'Filtracion' WHEN 3 THEN 'Desinfeccion'        WHEN 4 THEN 'Fangos' ELSE 'Bombeo'    END AS seccion,    'Documento técnico ' || SEQ4() || ' - Sección ' || CASE MOD(SEQ4(), 6) WHEN 0 THEN 'Pretratamiento' WHEN 1 THEN 'Coagulación' WHEN 2 THEN 'Filtración' WHEN 3 THEN 'Desinfección' WHEN 4 THEN 'Fangos' ELSE 'Bombeo' END AS titulo,    'Contenido técnico del documento ' || SEQ4() || '. Este procedimiento describe las operaciones de ' ||    CASE MOD(SEQ4(), 6) WHEN 0 THEN 'desbaste y desarenado del agua bruta, incluyendo limpieza de rejas y extracción de arenas'        WHEN 1 THEN 'adición de coagulante (sulfato de aluminio) y floculante (polielectrolito) para eliminar partículas en suspensión'        WHEN 2 THEN 'filtración rápida por arena y carbón activo para reducir turbidez por debajo de 1 NTU'        WHEN 3 THEN 'dosificación de hipoclorito sódico para garantizar cloro residual de 0.2-0.5 mg/l en red'        WHEN 4 THEN 'espesamiento, deshidratación y gestión de fangos generados en el proceso de tratamiento'        ELSE 'impulsión de agua tratada a depósitos de cabecera y regulación de presión en red de distribución'    END || '. Parámetros de control: caudal, presión, turbidez, pH, cloro residual. Frecuencia de revisión: diaria. Responsable: Jefe de Turno.' AS contenidoFROM TABLE(GENERATOR(ROWCOUNT => 80));SELECT COUNT(*) AS total_manuales, tipo, COUNT(*) AS por_tipo FROM manuales_planta GROUP BY tipo;

## Paso 3 — Generar Datos SCADA SimuladosSimulamos 500 lecturas de sensores de las últimas 24 horas: caudal, turbidez, pH, cloro, presión y nivel.

In [ ]:
CREATE OR REPLACE TABLE lecturas_scada ASSELECT    'SCADA-' || LPAD(SEQ4()::STRING, 5, '0') AS lectura_id,    DATEADD('minute', -SEQ4() * 3, CURRENT_TIMESTAMP()) AS timestamp,    CASE MOD(SEQ4(), 6)        WHEN 0 THEN 'Caudal_Entrada' WHEN 1 THEN 'Turbidez_Bruta'        WHEN 2 THEN 'pH_Bruto' WHEN 3 THEN 'Cloro_Residual'        WHEN 4 THEN 'Presion_Red' ELSE 'Nivel_Deposito'    END AS sensor,    ROUND(CASE MOD(SEQ4(), 6)        WHEN 0 THEN 150 + UNIFORM(-50, 100, RANDOM())        WHEN 1 THEN 5 + UNIFORM(0, 80, RANDOM()) * 0.1 + ABS(UNIFORM(-5, 5, RANDOM()))        WHEN 2 THEN 7.0 + UNIFORM(-5, 10, RANDOM()) * 0.1        WHEN 3 THEN 0.3 + UNIFORM(0, 15, RANDOM()) * 0.1        WHEN 4 THEN 3.0 + UNIFORM(-10, 15, RANDOM()) * 0.1        ELSE 60 + UNIFORM(-20, 30, RANDOM())    END, 2) AS valor,    CASE MOD(SEQ4(), 6)        WHEN 0 THEN 'm3/h' WHEN 1 THEN 'NTU' WHEN 2 THEN 'pH'        WHEN 3 THEN 'mg/l' WHEN 4 THEN 'bar' ELSE '%'    END AS unidad,    CASE MOD(SEQ4(), 3) WHEN 0 THEN 'Pretratamiento' WHEN 1 THEN 'Tratamiento' ELSE 'Distribucion' END AS zona_planta,    CASE WHEN UNIFORM(0, 100, RANDOM()) < 3 THEN 'Alarma'         WHEN UNIFORM(0, 100, RANDOM()) < 13 THEN 'Alerta'         ELSE 'Normal' END AS estadoFROM TABLE(GENERATOR(ROWCOUNT => 500));SELECT sensor, ROUND(AVG(valor),2) AS media, MIN(valor) AS min_val, MAX(valor) AS max_val, COUNT(*) AS lecturas FROM lecturas_scada GROUP BY sensor;

## Paso 4 — Crear Chunks para IndexaciónDividimos los documentos en fragmentos de ~500 caracteres para búsqueda semántica con Cortex Search.

In [ ]:
CREATE OR REPLACE TABLE chunks_planta ASSELECT    doc_id || '-' || chunk_orden AS chunk_id,    doc_id, tipo, seccion,    SUBSTR(contenido, (chunk_orden - 1) * 450 + 1, 500) AS chunk_texto,    chunk_ordenFROM manuales_planta,     LATERAL FLATTEN(ARRAY_GENERATE_RANGE(1, GREATEST(2, CEIL(LENGTH(contenido) / 450)::INT + 1))) AS t(chunk_orden);SELECT COUNT(*) AS total_chunks FROM chunks_planta;

## Paso 5 — Crear Cortex Search ServiceIndexamos los chunks para búsqueda semántica que permita al asistente encontrar la documentación relevante.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE buscador_planta  ON chunks_planta  WAREHOUSE = COMPUTE_WH  TARGET_LAG = '1 hour'  AS (    SELECT chunk_texto, chunk_id, doc_id, tipo, seccion    FROM chunks_planta  );

## Paso 6 — Implementar Pipeline RAGCreamos la función que combina búsqueda semántica + datos SCADA en tiempo real + generación de respuesta con Cortex AI.

In [ ]:
CREATE OR REPLACE FUNCTION preguntar_planta(pregunta STRING)RETURNS STRINGLANGUAGE SQLAS$$    SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',        'Eres un asistente técnico de planta de tratamiento de agua. ' ||        'Responde basándote SOLO en el contexto proporcionado. Cita las fuentes. ' ||        'Contexto documental: ' || (            SELECT LISTAGG(chunk_texto, ' | ') WITHIN GROUP (ORDER BY chunk_id)            FROM chunks_planta            WHERE CONTAINS(LOWER(chunk_texto), LOWER(SPLIT_PART(pregunta, ' ', 1)))            LIMIT 5        ) ||        ' Datos SCADA recientes: ' || (            SELECT LISTAGG(sensor || '=' || valor || unidad || ' (' || estado || ')', ', ')            FROM (SELECT * FROM lecturas_scada ORDER BY timestamp DESC LIMIT 10)        ) ||        ' Pregunta del operador: ' || pregunta    )$$;SELECT preguntar_planta('¿Cuál es la turbidez actual del agua bruta?') AS respuesta;

## Paso 7 — Dashboard Streamlit**Qué vamos a construir:** Interfaz conversacional para operadores con lecturas SCADA en sidebar y chat principal.

In [ ]:
import streamlit as stfrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()st.set_page_config(page_title="Talk to your Plant", layout="wide")st.title("🏭 Talk to your Plant — Asistente IA")with st.sidebar:    st.header("📊 SCADA en Tiempo Real")    scada = session.sql("SELECT sensor, valor, unidad, estado FROM lecturas_scada ORDER BY timestamp DESC LIMIT 12").to_pandas()    for _, row in scada.iterrows():        color = "🟢" if row["ESTADO"] == "Normal" else "🟡" if row["ESTADO"] == "Alerta" else "🔴"        st.metric(f"{color} {row['SENSOR']}", f"{row['VALOR']} {row['UNIDAD']}")st.markdown("---")pregunta = st.text_input("💬 Pregunta al asistente de planta:", placeholder="¿Cuál es el procedimiento si sube la turbidez?")if pregunta:    with st.spinner("Consultando..."):        resp = session.sql(f"SELECT preguntar_planta('{pregunta}') AS respuesta").collect()        st.success(resp[0]["RESPUESTA"])

## Paso 8 — Limpieza (Opcional)Descomenta las siguientes líneas para eliminar los objetos creados.

In [ ]:
-- DROP DATABASE IF EXISTS TALK_TO_PLANT_DB;-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;